In [ ]:
import xarray as xr
import matplotlib.pylab as plt
import xarray_regrid.methods.conservative  # pylint: disable=unused-import
from functools import partial
import xskillscore as xs
import pint_xarray
from dask.diagnostics import ProgressBar
import json
import pandas as pd
import numpy as np

from pism_terra.filtering import importance_sampling
from pism_terra.processing import preprocess_netcdf as preprocess

data_dir = "/Volumes/LHS800DATA"

pctls = [0.05, 0.95]
fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

debm_uq_vars = {'surface.debm_simple.c1': 'c1', 
                'surface.debm_simple.c2': 'c2',
                'surface.debm_simple.air_temp_all_precip_as_snow': 'as_snow', 
                'surface.debm_simple.air_temp_all_precip_as_rain': 'as_rain', 
                'surface.debm_simple.refreeze': 'refreeze'}

pdd_uq_vars = {'surface.pdd.factor_ice': 'fice', 
               'surface.pdd.factor_snow': 'fsnow',
               'surface.pdd.refreeze': 'refreeze'}

m_vars = ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]

obs = xr.open_dataset(f"{data_dir}/2026_04_kitp_debm_calib/kitp/input/v3/HIRHAM5-ERA5_YMM_1990_2019_v3.nc", engine="netcdf4",
                     decode_times=False,
                     decode_timedelta=False, chunks=None).drop_dims("nv", errors="ignore")

obs = obs.pint.quantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[v] = obs[v].pint.to("kg m^-2 yr^-1")
obs = obs.pint.dequantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[f"{v}_error"] = xr.where(obs[v] != 0, 0.10 * obs[v], 1e-8)

for ebm, ebm_uq_vars, fudge_factor in zip(["debm", "pdd"], [debm_uq_vars, pdd_uq_vars], [1000, 250]):
    
    ds = xr.open_mfdataset(f"{data_dir}/2026_04_kitp_{ebm}_calib/output/spatial/clipped_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_*.nc",
                         preprocess=partial(preprocess,
                                            uq_regexp=None,
                                            exp_regexp="uq_(.+?)_"),
                        engine="netcdf4",
                        join="outer",
                        compat="no_conflicts",
                        parallel=True,
                        chunks="auto",
                        decode_times=False,
                        decode_timedelta=False).drop_dims("nv", errors="ignore")

    
    ebm_uq_df = ds.pism_config.to_series().apply(json.loads).apply(pd.Series)[ebm_uq_vars.keys()]
    ds["time"] = _obs["time"]

    _obs = obs.regrid.conservative(ds.drop_vars("pism_config")).squeeze()
    mask = ds[m_vars].isel(exp_id=0).notnull()                                                                                                                                         
    _obs[m_vars] = _obs[m_vars].where(mask) 
    
    for v in m_vars:
    
        with ProgressBar():
            ebm_filtered = importance_sampling(ds, _obs, 
                            sim_var=v,
                            obs_mean_var=v, 
                            obs_std_var=f"{v}_error",
                            sum_dims=['time', 'x', 'y'],
                            n_samples=ds.exp_id.size,
                            fudge_factor=fudge_factor
                                              )

            ebm_sampled_ids = ebm_filtered.exp_id_sampled.values
            ebm_counts = pd.Series(ebm_sampled_ids).value_counts()
                                                                                                                                                                                                 
            # Reindex config_df to the sampled IDs and plot histograms                                                                                                                         
            ds_sampled_configs = ebm_uq_df.loc[ebm_counts.index].reindex(ebm_counts.index)                                                                                                                
            most_sampled_id = ebm_counts.idxmax()                                                                                                                                  
            most_sampled_params = ebm_uq_df.loc[most_sampled_id]
            print(f"\n{ebm} / {v} — most sampled id={most_sampled_id} (count={ebm_counts.max()})")                                                                                 
            for k, short in ebm_uq_vars.items():                                                                                                                                   
                print(f"  {short}: {most_sampled_params[k]:.6g}")     
                  
            fig, axes = plt.subplots(1, len(ebm_uq_vars), sharey=True, figsize=(6.4, 1.8))           
            for ax, (key, value) in zip(axes.flat, ebm_uq_vars.items()):                                                                                                                                             
                    # Repeat each parameter value by its sample count                                                                                                                              
                      values = np.repeat(ebm_uq_df[key].values,                                                                                                                                      
                                     ebm_counts.reindex(ebm_uq_df.index, fill_value=0).values.astype(int))                                                                                           
                      ax.hist(values, bins=15)                                                                                                                                                       
                      ax.set_xlabel(value)    
                      ax.set_xlim(ebm_uq_df[key].min(), ebm_uq_df[key].max())
                      # ax.set_ylabel("Count")                                                                                                                                                         
            fig.tight_layout()   
            fig.savefig(f"{ebm}_{v}.png", dpi=300)
            plt.close()
            del fig
            
            
            # Plot smallest RMSE                                                                                                                                                       
            rmse = xs.rmse(ds[v], _obs[v], dim=["time", "x", "y"], skipna=True).compute()
            best_id = rmse.idxmin(dim="exp_id").values                                                                                                                                                                                                                                                                                                                          
            sim_best = ds[v].sel(exp_id=best_id).mean(dim="time").squeeze()                                                                                                            
            obs_mean = _obs[v].mean(dim="time").squeeze()                                                                                                                               
            vmin = min(float(obs_mean.min()), float(sim_best.min()))
            vmax = max(float(obs_mean.max()), float(sim_best.max()))
            best_params = ebm_uq_df.loc[best_id]                                                                                                                                                                                         
            fig, axes = plt.subplots(1, 3, sharey=True, figsize=(12, 4))                                                                                                                         
            obs_mean.plot(ax=axes[0], vmin=vmin, vmax=vmax)                                                                                                                            
            axes[0].set_title("Observed")                                                                                                                                              
            sim_best.plot(ax=axes[1], vmin=vmin, vmax=vmax)                                                                                                                            
            param_str = ", ".join(f"{v}={best_params[k]:.4g}" for k, v in ebm_uq_vars.items())                                                                                                 
            axes[1].set_title(f"Best (id={best_id}, RMSE={float(rmse.sel(exp_id=best_id)):.1f})\n{param_str}") 
            (sim_best - obs_mean).plot(ax=axes[2], cmap="RdBu")                                                                                                                        
            axes[2].set_title("Difference")                   
            fig.tight_layout()                                                                                                                                                         
            fig.savefig(f"{ebm}_{v}_best_rmse.png", dpi=300)                                                                                                                           
            plt.close()
            del fig 

In [ ]:
ds.climatic_mass_balance.mean(dim=["time"]).isel(exp_id=1).plot()

In [ ]:
var

In [ ]:
sampled_ids = debm_filtered.exp_id_sampled.values
counts = pd.Series(sampled_ids).value_counts()
                                                                                                                                                                                     
# Reindex config_df to the sampled IDs and plot histograms                                                                                                                         
sampled_configs = debm_uq_df.loc[counts.index].reindex(counts.index)                                                                                                                
                                                                                                                                                                                     
fig, axes = plt.subplots(2, 3, figsize=(12, 4))           
for ax, var in zip(axes.flat, debm_uq_vars):                                                                                                                                             
        # Repeat each parameter value by its sample count                                                                                                                              
          values = np.repeat(debm_uq_df[var].values,                                                                                                                                      
                         counts.reindex(debm_uq_df.index, fill_value=0).values.astype(int))                                                                                           
          ax.hist(values, bins=15)                                                                                                                                                       
          ax.set_xlabel(var)                                    
          ax.set_ylabel("Count")                                                                                                                                                         
plt.tight_layout()          

In [ ]:
_debm_filtered.exp_id_sampled

In [ ]:
df = pdd.pism_config.to_series().apply(json.loads).apply(pd.Series)[pdd_uq_vars]

In [ ]:
import pandas as pd

In [ ]:
ds.exp_id.size

In [ ]:
_obs.climatic_mass_balance.mean(dim=["x", "y"]).plot()

In [ ]:
ds = xr.open_dataset("/Volumes/LHS800DATA/2026_04_kitp_debm/output/spatial/fldsum_spatial_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc")
ds_scalar = xr.open_dataset("/Volumes/LHS800DATA/2026_04_kitp_debm/output/scalar/scalar_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc")
ds_scalar = ds_scalar.resample(time="YE").mean()

ds_new = xr.open_dataset("/Volumes/LHS800DATA/2026_04_kitp_debm_strong/output/spatial/fldsum_spatial_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc")
ds_new = ds_new.sel(basin="GIS")
#ds_new = ds_new.resample(time="YE").mean()


fig, ax = plt.subplots(1, 1)
#ds.sel(basin="GIS").tendency_of_ice_mass.plot(ax=ax)
#ds_scalar.tendency_of_ice_mass.plot(ax=ax)
ds_new.tendency_of_ice_mass.plot(ax=ax)



In [ ]:
ds_scalar.tendency_of_ice_mass.plot(ax=ax)

In [ ]:
ds_scalar.tendency_of_ice_mass.plot()

In [ ]:
ds_sampled_configs